# 13 · 正则化：Dropout 与 L2

> **本节属于 Part 5 · 优化与训练工程。**

当模型容量很大、数据又带噪声时，它会**过拟合**——连训练数据里的噪声都"背"下来，导致在新数据上表现变差。本节我们实现两种最常用的"刹车"：**L2 正则（权重衰减）**与 **Dropout**，并在一个含噪的二维数据集上**直观看到**它们如何让决策边界更平滑、泛化更好。

## 学习目标

- 直观理解**过拟合**与正则化
- 用优化器的 `weight_decay` 实现 **L2 正则**
- 实现并使用 `Dropout`（含训练/推理两种模式）
- 通过**决策边界可视化**对比加/不加正则的泛化差异

## 制造一个容易过拟合的场景

用一个**含较大噪声**的"双月牙"数据集，训练点只有 120 个，却用一个偏大的网络（两层 64 维隐藏层）。不加约束的话，它会去拟合那些噪声点。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import minitorch
from minitorch import Tensor, nn, no_grad, data
from minitorch.optim import Adam

def make_moons(n, noise, seed):
    rng = np.random.RandomState(seed)
    n0, n1 = n // 2, n - n // 2
    t0, t1 = np.linspace(0, np.pi, n0), np.linspace(0, np.pi, n1)
    X = np.vstack([np.c_[np.cos(t0), np.sin(t0)],
                   np.c_[1 - np.cos(t1), 1 - np.sin(t1) - 0.5]]) + rng.randn(n, 2) * noise
    y = np.array([0] * n0 + [1] * n1)
    return X, y

X_tr, y_tr = make_moons(120, noise=0.35, seed=0)     # 训练集（含噪、量少）
X_te, y_te = make_moons(2000, noise=0.35, seed=1)    # 测试集（同分布、量大）
print("训练点:", X_tr.shape[0], " 测试点:", X_te.shape[0])

def build(dropout=0.0):
    L = [nn.Linear(2, 64), nn.ReLU()]
    if dropout: L.append(nn.Dropout(dropout))
    L += [nn.Linear(64, 64), nn.ReLU()]
    if dropout: L.append(nn.Dropout(dropout))
    L += [nn.Linear(64, 2)]
    return nn.Sequential(*L)

def train(model, weight_decay=0.0, epochs=300):
    opt = Adam(model.parameters(), lr=3e-3, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    loader = data.DataLoader(data.TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(Tensor(xb)), yb).backward()
            opt.step()
    model.eval()
    with no_grad():
        tr = (model(Tensor(X_tr)).data.argmax(1) == y_tr).mean()
        te = (model(Tensor(X_te)).data.argmax(1) == y_te).mean()
    return model, tr, te

## 三种设置对比

In [ ]:
minitorch.set_seed(0); m_plain, tr0, te0 = train(build(), weight_decay=0.0)
minitorch.set_seed(0); m_l2,    tr1, te1 = train(build(), weight_decay=1e-2)
minitorch.set_seed(0); m_drop,  tr2, te2 = train(build(0.3), weight_decay=5e-3)

print(f"无正则      ：train {tr0*100:5.1f}%  test {te0*100:5.1f}%  gap {(tr0-te0)*100:+.1f}%")
print(f"L2(1e-2)    ：train {tr1*100:5.1f}%  test {te1*100:5.1f}%  gap {(tr1-te1)*100:+.1f}%")
print(f"Dropout+L2  ：train {tr2*100:5.1f}%  test {te2*100:5.1f}%  gap {(tr2-te2)*100:+.1f}%")

无正则的模型训练准确率更高，但**测试准确率反而更低**——它把噪声也学进去了。加上正则后，训练准确率略降，但**测试准确率上升、训练/测试差距缩小**。

## 决策边界：一图胜千言

把三个模型的决策边界画出来。无正则的边界为了迁就个别噪声点会变得**扭曲**；正则化让边界更**平滑**，也就更接近数据真正的结构。

In [ ]:
def plot_boundary(ax, model, title):
    model.eval()
    xx, yy = np.meshgrid(np.linspace(-1.8, 2.8, 200), np.linspace(-1.3, 1.8, 200))
    with no_grad():
        Z = model(Tensor(np.c_[xx.ravel(), yy.ravel()])).data.argmax(1).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=1, cmap="bwr", alpha=0.25)
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="bwr", s=16, edgecolors="k", linewidths=0.3)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
plot_boundary(ax[0], m_plain, f"No reg (test {te0*100:.1f}%)")
plot_boundary(ax[1], m_l2,    f"L2 (test {te1*100:.1f}%)")
plot_boundary(ax[2], m_drop,  f"Dropout+L2 (test {te2*100:.1f}%)")
plt.tight_layout(); plt.show()

## Dropout 的实现要点

**inverted dropout**：训练时以概率 $p$ 把激活置零，并把保留的部分除以 $(1-p)$；推理时**原样输出**。因此必须区分 `train()` / `eval()` 模式（`Module` 已统一管理）。

In [ ]:
import inspect
print(inspect.getsource(nn.Dropout.forward))

d = nn.Dropout(0.5)
x = Tensor(np.ones(10))
d.train(); print("train 模式（约一半置零，其余×2）:", d(x).data)
d.eval();  print("eval  模式（原样）:", d(x).data)

## 📦 沉淀进 minitorch

`Dropout` 在 `minitorch/nn/dropout.py`；**L2 正则**由优化器的 `weight_decay` 实现（见 `SGD/Adam`，对应 PyTorch 的 `weight_decay`）。

## 小练习

1. **正则强度**：把 `weight_decay` 从 `0` 扫到 `1e-1`，观察决策边界从"扭曲"到"过度平滑（欠拟合）"的全过程。
2. **只用 Dropout**：去掉 L2、只保留 `Dropout(0.3)`，泛化提升多少？换 `Dropout(0.6)` 呢？
3. **更多数据**：把训练点从 120 提到 2000，过拟合是否自然消失？（数据本身就是最好的正则。）

## 小结 & 下一站

✅ 我们用 L2 与 Dropout 抑制了过拟合，并通过决策边界直观看到了"平滑 = 更好的泛化"。`train()/eval()` 模式的重要性也再次得到体现。

**下一站 → `14_batchnorm_layernorm`**：实现 **BatchNorm 与 LayerNorm**——它们能显著加速并稳定训练。我们会**手推 BatchNorm 那个著名的反向公式**，再用 gradcheck 证明 autograd 自动算出的结果与它一致。